# Training PPO - Coordinación de Agentes para Mitigación de Slippage (IPSA)

**Sprint 4 - Tarea 3.1.1/3.1.2/3.1.3/3.1.4/3.1.5**

Responsable: Paolo Sepúlveda (PS)

Fecha: Septiembre 2026

## 1. Setup & Dependencias

In [ ]:
# Instalar dependencias (solo primera vez)
!pip install gymnasium torch numpy scipy pandas matplotlib seaborn scikit-learn tqdm

In [ ]:
# Clonar repo si es necesario
import os
import sys

if not os.path.exists('/content/slippage'):
    !git clone https://github.com/Vatelfil/slippage.git /content/slippage

sys.path.insert(0, '/content/slippage')
os.chdir('/content/slippage')
print("✓ Repositorio listo")

In [ ]:
# Imports
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Normal, Categorical
import gymnasium as gym
from collections import deque
import json
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Imports del proyecto
from src.envs.spaces import MaestroSpace, EjecutorSpace, MaestroActionSpace, EjecutorActionSpace
from src.envs.maestro_env import MaestroEnv
from src.envs.ejecutor_env import EjecutorEnv
from src.models.actor_critic import MasterActorCritic, ExecutorActorCritic

print("✓ Dependencias importadas")

## 2. Configuración de Hiperparámetros

In [ ]:
# Hiperparámetros PPO
config = {
    # Arquitectura
    'hidden_dim': 128,
    'num_layers': 2,
    'activation': 'relu',

    # Training
    'learning_rate': 3e-4,
    'gamma': 0.99,  # descuento del Maestro (MDP_M, Título I sección 4: gamma_M=0.99)
    'gamma_ejecutor': 0.95,  # descuento de los Ejecutores (MDP_E, Título I sección 5: gamma_E=0.95)
    'gae_lambda': 0.95,  # GAE
    'clip_ratio': 0.2,  # PPO clip
    'entropy_coef': 0.01,
    'value_coef': 0.5,

    # Rollout
    'max_steps': 30,  # pasos por episodio (30 min simulados)
    'num_envs': 4,  # ambientes paralelos -- NO usado aún por el training loop de referencia
                    # (Sección 11): ese loop corre 1 env por agente en serie, no vectorizado.
                    # Vectorizar (ej. gymnasium.vector) queda pendiente para cuando ABIDES-Gym
                    # esté integrado y el costo por step lo justifique.
    'rollout_steps': 2048,  # samples antes de update
    'epochs_per_update': 10,

    # Training loop
    'total_timesteps': 1_000_000,
    'eval_interval': 50_000,
    'checkpoint_interval': 100_000,

    # Data
    'batch_size': 64,
    'seed': 42,
}

print(json.dumps(config, indent=2))


## 3. Redes Actor-Critic (Base)

In [ ]:
# Las redes reales viven en src/models/actor_critic.py (Mauricio, tareas 2.1.1/2.1.2)
# y ya están importadas en la celda de imports (MasterActorCritic, ExecutorActorCritic).
#
# (23 sept, PS) Se agregó `get_action_and_value(obs, action=None)` a `BaseActorCritic`
# en ese mismo archivo -- ambas redes son de acción discreta única (un solo
# actor_head), así que Categorical alcanza para las dos. Antes esta celda
# definía una clase `ActorCritic` local (placeholder) que nunca estuvo
# conectada a las redes reales; se eliminó para no tener dos implementaciones
# divergentes.

print("✓ Usando MasterActorCritic / ExecutorActorCritic de src/models/actor_critic.py")
print("  (con get_action_and_value() agregado para el rollout PPO)")


## 4. Inicializar Ambientes

In [ ]:
# Maestro
maestro_env = MaestroEnv(max_steps=config['max_steps'])
maestro_obs_dim = maestro_env.observation_space.shape[0]

# ✅ Resuelto (23 sept, PS): MaestroEnv.action_space ahora es Discrete(40)
# (antes era Box continuo, incompatible con MasterActorCritic). Ver
# spaces.MaestroDiscreteActionSpace para el decode idx -> (alpha_t, ventana_min).
maestro_action_dim = maestro_env.action_space.n  # 40

print(f"Maestro:")
print(f"  Obs space: {maestro_obs_dim}D")
print(f"  Action space: {maestro_env.action_space} (idx -> alpha_t, ventana_min vía env.step())")

# Ejecutores
ejecutor_envs = [EjecutorEnv(executor_id=i, max_steps=config['max_steps']) for i in range(3)]
ejecutor_obs_dim = ejecutor_envs[0].observation_space.shape[0]

# EjecutorActionSpace es MultiDiscrete([3, 10, 8]) = order_type x volume_bucket x price_level
# (ver src/envs/spaces.py). ExecutorActorCritic tiene un único actor_head discreto,
# así que se usa la acción "aplanada": 3*10*8 = 240 acciones posibles.
EJECUTOR_NVEC = ejecutor_envs[0].action_space.nvec  # [3, 10, 8]
ejecutor_action_dim = int(EJECUTOR_NVEC.prod())  # 240

print(f"\nEjecutores (x3):")
print(f"  Obs space: {ejecutor_obs_dim}D")
print(f"  Env action_space: MultiDiscrete({list(EJECUTOR_NVEC)})")
print(f"  Action dim de la red (aplanado): {ejecutor_action_dim}")
print(f"  ✅ Resuelto (23 sept, PS): usar ejecutor_envs[i]._action_space_wrapper.decode_flat(idx)")
print(f"    para convertir el índice 0-239 de la red al array [order_type, volume_bucket,")
print(f"    price_level] que espera EjecutorEnv.step(). Verificado round-trip para los 240 valores.")


## 5. Inicializar Redes (usar las de Mauricio cuando estén listas)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Maestro: MasterActorCritic fija internamente action_dim=40 (ver clase), solo recibe obs_dim.
maestro_actor_critic = MasterActorCritic(obs_dim=maestro_obs_dim).to(device)
maestro_optimizer = optim.Adam(maestro_actor_critic.parameters(), lr=config['learning_rate'])

# Ejecutores: 3 redes independientes (Apertura / Media jornada / Cierre), cada una con
# su propio optimizer (no comparten pesos -- ver arquitectura CTDE del Título I, 3.1).
ejecutor_actor_critics = [
    ExecutorActorCritic(obs_dim=ejecutor_obs_dim, action_dim=ejecutor_action_dim).to(device)
    for _ in range(3)
]
ejecutor_optimizers = [optim.Adam(ac.parameters(), lr=config['learning_rate']) for ac in ejecutor_actor_critics]

print("✓ Redes reales inicializadas (src/models/actor_critic.py)")
print(f"  Maestro: obs_dim={maestro_obs_dim}, action_dim=40 (fijo en MasterActorCritic)")
print(f"  Ejecutores (x3): obs_dim={ejecutor_obs_dim}, action_dim={ejecutor_action_dim}")


## 6. Buffer de Rollout (Experiencias)

In [ ]:
class RolloutBuffer:
    """Buffer para almacenar experiencias del rollout."""
    
    def __init__(self, capacity):
        self.capacity = capacity
        self.clear()
    
    def clear(self):
        self.obs = []
        self.actions = []
        self.log_probs = []
        self.rewards = []
        self.values = []
        self.dones = []
        self.returns = []
        self.advantages = []
    
    def add(self, obs, action, log_prob, reward, value, done):
        self.obs.append(obs)
        self.actions.append(action)
        self.log_probs.append(log_prob)
        self.rewards.append(reward)
        self.values.append(value)
        self.dones.append(done)
    
    def compute_returns_advantages(self, gamma, gae_lambda):
        """Compute GAE advantages and returns."""
        self.returns = []
        self.advantages = []
        
        gae = 0
        for t in reversed(range(len(self.rewards))):
            if t == len(self.rewards) - 1:
                next_value = 0
            else:
                next_value = self.values[t + 1]
            
            delta = self.rewards[t] + gamma * next_value - self.values[t]
            gae = delta + gamma * gae_lambda * gae
            
            self.advantages.insert(0, gae)
            self.returns.insert(0, gae + self.values[t])
    
    def get_batches(self, batch_size):
        """Yield minibatches."""
        n = len(self.obs)
        indices = np.random.permutation(n)
        
        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)
            batch_indices = indices[start:end]
            
            yield {
                'obs': torch.FloatTensor(np.array([self.obs[i] for i in batch_indices])).to(device),
                'actions': torch.LongTensor([self.actions[i] for i in batch_indices]).to(device),
                'log_probs': torch.FloatTensor([self.log_probs[i] for i in batch_indices]).to(device),
                'returns': torch.FloatTensor([self.returns[i] for i in batch_indices]).to(device),
                'advantages': torch.FloatTensor([self.advantages[i] for i in batch_indices]).to(device),
            }

print("✓ RolloutBuffer definido")

## 7. Training Loop PPO (Skeleton)

In [ ]:
# ============================================================
# BORRADOR DE REFERENCIA (PS, 23 sept) — Tarea 3.1.1, dueño formal: Mauricio.
# Implementación correcta del objetivo PPO clipeado (L_CLIP) + value loss +
# entropy bonus, pensada para que Mauricio la revise/ajuste, no como entrega
# oficial de su tarea. Ver DIAGNOSTICO_Y_PLAN_23SEP.md.
# ============================================================

def ppo_update(actor_critic, optimizer, batch, clip_ratio, entropy_coef, value_coef,
                max_grad_norm=0.5):
    """Actualiza la red con el objetivo PPO clipeado.

    L_CLIP(θ) = E_t[ min(r_t(θ)·A_t, clip(r_t(θ), 1−ε, 1+ε)·A_t) ]
    r_t(θ) = π_θ(a_t|s_t) / π_θ_old(a_t|s_t)  (aquí: exp(new_logp - old_logp))

    Args:
        actor_critic: red con `.get_action_and_value(obs, action)` (ver
            src/models/actor_critic.py — método agregado el 23 sept).
        batch: dict de `RolloutBuffer.get_batches()`: 'obs', 'actions',
            'log_probs' (viejos, de cuando se recolectó el rollout),
            'returns', 'advantages'.
    """
    obs = batch['obs']
    actions = batch['actions']
    old_log_probs = batch['log_probs']
    returns = batch['returns']
    advantages = batch['advantages']

    # Normalizar advantages (práctica estándar PPO — reduce varianza del gradiente)
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    _, new_log_probs, entropy, values = actor_critic.get_action_and_value(obs, actions)
    values = values.squeeze(-1)

    ratio = torch.exp(new_log_probs - old_log_probs)
    surr1 = ratio * advantages
    surr2 = torch.clamp(ratio, 1 - clip_ratio, 1 + clip_ratio) * advantages
    policy_loss = -torch.min(surr1, surr2).mean()

    value_loss = F.mse_loss(values, returns)
    entropy_bonus = entropy.mean()

    loss = policy_loss + value_coef * value_loss - entropy_coef * entropy_bonus

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(actor_critic.parameters(), max_grad_norm)
    optimizer.step()

    return {
        'policy_loss': policy_loss.item(),
        'value_loss': value_loss.item(),
        'entropy': entropy_bonus.item(),
    }


def run_ppo_epochs(actor_critic, optimizer, buffer, epochs, batch_size,
                    clip_ratio, entropy_coef, value_coef):
    """Corre `epochs` pasadas de PPO sobre el buffer ya con GAE calculado.

    Returns:
        dict con el promedio de policy_loss/value_loss/entropy sobre todas
        las minibatches de todas las épocas (lo que consume log_metrics()).
    """
    accum = {'policy_loss': 0.0, 'value_loss': 0.0, 'entropy': 0.0}
    n_batches = 0
    for _ in range(epochs):
        for batch in buffer.get_batches(batch_size):
            losses = ppo_update(actor_critic, optimizer, batch, clip_ratio, entropy_coef, value_coef)
            for k in accum:
                accum[k] += losses[k]
            n_batches += 1
    if n_batches == 0:
        return {'policy_loss': 0.0, 'value_loss': 0.0, 'entropy': 0.0}
    return {k: v / n_batches for k, v in accum.items()}


# Tracking
train_stats = {
    'maestro_return': deque(maxlen=100),
    'ejecutor_return': [deque(maxlen=100) for _ in range(3)],
    'losses': [],
}

# Buffers de rollout: uno para el Maestro, uno por cada uno de los 3 Ejecutores.
maestro_buffer = RolloutBuffer(capacity=config['rollout_steps'])
ejecutor_buffers = [RolloutBuffer(capacity=config['rollout_steps']) for _ in range(3)]

print("✓ ppo_update() implementado (L_CLIP + value loss + entropy) — borrador para revisión de Mauricio")
print("✓ Buffers de rollout inicializados (Maestro + 3 Ejecutores)")


## 8. Checkpointing & Logging

In [ ]:
import os
import csv
from datetime import datetime
from torch.utils.tensorboard import SummaryWriter

# ============================================================
# 8.1 Setup de logging (TensorBoard + CSV) y checkpoints
# ============================================================
# Todo queda bajo una misma carpeta run, timestamped, para que cada
# corrida de Colab no pise los logs de la anterior.

run_id = f"ppo_sprint4_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
log_dir = Path('logs') / run_id
checkpoint_dir = Path('checkpoints') / run_id
log_dir.mkdir(parents=True, exist_ok=True)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

writer = SummaryWriter(log_dir=str(log_dir))
csv_path = log_dir / 'metrics.csv'
CSV_COLUMNS = [
    'step',
    'maestro_return', 'maestro_loss_policy', 'maestro_loss_value', 'maestro_entropy',
    'ejecutor0_return', 'ejecutor1_return', 'ejecutor2_return',
]

# Crear el CSV con encabezado si todavia no existe (permite resumir una corrida
# sin duplicar el header si el notebook se re-ejecuta sobre el mismo run_id).
if not csv_path.exists():
    with open(csv_path, 'w', newline='') as f:
        csv.writer(f).writerow(CSV_COLUMNS)

print(f"Run ID: {run_id}")
print(f"Logs (TensorBoard + CSV): {log_dir}")
print(f"Checkpoints: {checkpoint_dir}")


# ============================================================
# 8.2 Logging de hiperparametros (una sola vez, al inicio del run)
# ============================================================

def log_config(config: dict, log_dir: Path = log_dir):
    """Guarda config.json y lo deja tambien legible en TensorBoard (pestaña TEXT)."""
    config_path = log_dir / 'config.json'
    with open(config_path, 'w') as f:
        json.dump(config, f, indent=2)
    writer.add_text('config', f"```json\n{json.dumps(config, indent=2)}\n```")
    print(f"✓ Config guardado en {config_path}")


log_config(config)


# ============================================================
# 8.3 log_metrics(): llamar cada N steps dentro del training loop
# ============================================================
# maestro_loss y ejecutor_losses siguen el formato que retorna ppo_update():
#   {'policy_loss': float, 'value_loss': float, 'entropy': float}

def log_metrics(step: int, maestro_return: float, maestro_loss: dict,
                 ejecutor_returns: list, ejecutor_losses: list = None):
    """
    Registra métricas de un step de entrenamiento en TensorBoard y en el CSV.

    Args:
        step: timestep global de entrenamiento (no de episodio).
        maestro_return: retorno acumulado del Maestro en el episodio/rollout.
        maestro_loss: dict con 'policy_loss', 'value_loss', 'entropy' (salida de ppo_update).
        ejecutor_returns: lista de 3 floats, uno por Ejecutor (Apertura/Media/Cierre).
        ejecutor_losses: lista opcional de 3 dicts (mismo formato que maestro_loss).
    """
    try:
        # --- TensorBoard ---
        writer.add_scalar('maestro/return', maestro_return, step)
        writer.add_scalar('maestro/loss/policy', maestro_loss['policy_loss'], step)
        writer.add_scalar('maestro/loss/value', maestro_loss['value_loss'], step)
        writer.add_scalar('maestro/entropy', maestro_loss['entropy'], step)

        for i, ret in enumerate(ejecutor_returns):
            writer.add_scalar(f'ejecutor{i}/return', ret, step)
            if ejecutor_losses is not None:
                writer.add_scalar(f'ejecutor{i}/loss/policy', ejecutor_losses[i]['policy_loss'], step)
                writer.add_scalar(f'ejecutor{i}/loss/value', ejecutor_losses[i]['value_loss'], step)
                writer.add_scalar(f'ejecutor{i}/entropy', ejecutor_losses[i]['entropy'], step)

        # --- CSV (para el análisis posterior con ConvergenceAnalyzer) ---
        row = [
            step, maestro_return,
            maestro_loss['policy_loss'], maestro_loss['value_loss'], maestro_loss['entropy'],
            *ejecutor_returns,
        ]
        with open(csv_path, 'a', newline='') as f:
            csv.writer(f).writerow(row)

        # --- Tracking en memoria (para plot_convergence() en esta misma notebook) ---
        train_stats['maestro_return'].append(maestro_return)
        for i, ret in enumerate(ejecutor_returns):
            train_stats['ejecutor_return'][i].append(ret)
        train_stats['losses'].append({'step': step, 'maestro': maestro_loss})

    except KeyError as e:
        # Si ppo_update todavia es el skeleton (Sección 7), maestro_loss puede no
        # traer todas las llaves esperadas: fallar de forma explícita y clara.
        raise KeyError(
            f"log_metrics: falta la llave {e} en maestro_loss/ejecutor_losses. "
            "Verifica que ppo_update() retorne 'policy_loss', 'value_loss' y 'entropy'."
        ) from e


print("✓ log_metrics() y log_config() definidos")


# ============================================================
# 8.4 Checkpointing: guardar y cargar estado completo del training
# ============================================================

def save_checkpoint(step: int):
    """
    Guarda pesos + optimizers + config + métricas acumuladas hasta 'step'.
    Permite resumir un training interrumpido (ej. Colab se desconecta).
    """
    checkpoint = {
        'step': step,
        'timestamp': datetime.now().isoformat(),
        'run_id': run_id,

        # Redes y optimizers
        'maestro_state_dict': maestro_actor_critic.state_dict(),
        'maestro_optimizer': maestro_optimizer.state_dict(),
        'ejecutor_state_dicts': [ac.state_dict() for ac in ejecutor_actor_critics],
        'ejecutor_optimizers': [opt.state_dict() for opt in ejecutor_optimizers],

        # Config, para reproducibilidad exacta de la corrida
        'config': config,

        # Métricas acumuladas hasta ahora (para reanudar gráficos sin recalcular)
        'train_stats': {
            'maestro_return': list(train_stats['maestro_return']),
            'ejecutor_return': [list(r) for r in train_stats['ejecutor_return']],
            'losses': train_stats['losses'],
        },
    }

    checkpoint_path = checkpoint_dir / f'checkpoint_{step:07d}.pt'
    torch.save(checkpoint, checkpoint_path)
    print(f"✓ Checkpoint guardado: {checkpoint_path}")
    return checkpoint_path


def load_checkpoint(path):
    """
    Carga un checkpoint y restaura pesos/optimizers/train_stats in-place.
    Requiere que maestro_actor_critic, ejecutor_actor_critics, maestro_optimizer
    y ejecutor_optimizers ya existan (mismas arquitecturas que al guardar).

    Returns:
        step (int): el timestep desde el cual continuar el training loop.
    """
    checkpoint = torch.load(path, map_location=device)

    maestro_actor_critic.load_state_dict(checkpoint['maestro_state_dict'])
    maestro_optimizer.load_state_dict(checkpoint['maestro_optimizer'])
    for ac, sd in zip(ejecutor_actor_critics, checkpoint['ejecutor_state_dicts']):
        ac.load_state_dict(sd)
    for opt, sd in zip(ejecutor_optimizers, checkpoint['ejecutor_optimizers']):
        opt.load_state_dict(sd)

    restored = checkpoint['train_stats']
    train_stats['maestro_return'] = deque(restored['maestro_return'], maxlen=100)
    train_stats['ejecutor_return'] = [deque(r, maxlen=100) for r in restored['ejecutor_return']]
    train_stats['losses'] = restored['losses']

    print(f"✓ Checkpoint cargado desde {path} (step={checkpoint['step']}, run_id={checkpoint.get('run_id', 'n/a')})")
    return checkpoint['step']


print("✓ save_checkpoint() / load_checkpoint() definidos")
print("\nUso dentro del training loop (Sección 11):")
print("  if step % config['eval_interval'] == 0:")
print("      log_metrics(step, R_m, L_m, Rs_e, Ls_e)")
print("  if step % config['checkpoint_interval'] == 0:")
print("      save_checkpoint(step)")


## 9. Validación (Evaluation)

In [ ]:
def evaluate(n_episodes=5):
    """Evaluar política sin exploración."""
    
    maestro_returns = []
    ejecutor_returns = [[] for _ in range(3)]
    
    for ep in range(n_episodes):
        # Reset
        maestro_obs, _ = maestro_env.reset()
        ejecutor_obss = [env.reset()[0] for env in ejecutor_envs]
        
        maestro_ep_return = 0
        ejecutor_ep_returns = [0] * 3
        done = False
        
        while not done:
            # Maestro step
            # TODO: step maestro
            
            # Ejecutores step
            for i in range(3):
                # TODO: step ejecutor i
                pass
        
        maestro_returns.append(maestro_ep_return)
        for i in range(3):
            ejecutor_returns[i].append(ejecutor_ep_returns[i])
    
    return {
        'maestro_mean': np.mean(maestro_returns),
        'ejecutor_means': [np.mean(r) for r in ejecutor_returns],
    }

print("✓ Evaluation function definida")

## 10. Análisis de Convergencia

In [ ]:
def plot_convergence():
    """
    Plotea las métricas acumuladas en train_stats durante esta sesión de Colab.

    Para análisis más completo (media móvil, test estadístico de tendencia,
    comparación entre runs guardados en logs/*/metrics.csv), usar
    src/analysis/convergence_analysis.py y el notebook Analysis_Sprint4.ipynb.
    """
    if len(train_stats['maestro_return']) == 0:
        print("⚠ train_stats está vacío todavía: corre algunos steps de log_metrics() antes de plotear.")
        return

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    # 1. Return del Maestro
    axes[0, 0].plot(list(train_stats['maestro_return']))
    axes[0, 0].set_title('Return - Maestro')
    axes[0, 0].set_xlabel('Log entry (cada eval_interval steps)')
    axes[0, 0].set_ylabel('Return')
    axes[0, 0].grid(True, alpha=0.3)

    # 2. Return de los 3 Ejecutores
    for i, ret_deque in enumerate(train_stats['ejecutor_return']):
        if len(ret_deque) > 0:
            axes[0, 1].plot(list(ret_deque), label=f'Ejecutor {i}')
    axes[0, 1].set_title('Return - Ejecutores')
    axes[0, 1].set_xlabel('Log entry')
    axes[0, 1].set_ylabel('Return')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # 3. Losses del Maestro (policy / value)
    if len(train_stats['losses']) > 0:
        policy_losses = [l['maestro']['policy_loss'] for l in train_stats['losses']]
        value_losses = [l['maestro']['value_loss'] for l in train_stats['losses']]
        axes[1, 0].plot(policy_losses, label='Policy loss')
        axes[1, 0].plot(value_losses, label='Value loss')
        axes[1, 0].set_title('Losses - Maestro')
        axes[1, 0].set_xlabel('Log entry')
        axes[1, 0].set_ylabel('Loss')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)

        # 4. Entropy del Maestro (debe decrecer si la política converge)
        entropies = [l['maestro']['entropy'] for l in train_stats['losses']]
        axes[1, 1].plot(entropies, color='purple')
        axes[1, 1].set_title('Entropy - Maestro')
        axes[1, 1].set_xlabel('Log entry')
        axes[1, 1].set_ylabel('Entropy')
        axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plot_path = checkpoint_dir / 'convergence.png'
    plt.savefig(plot_path, dpi=100)
    print(f"✓ Gráfico guardado: {plot_path}")
    plt.show()


print("✓ Análisis de convergencia definido")


## 11. MAIN: Training Loop (Cuando esté listo)

In [ ]:
# ============================================================
# BORRADOR DE REFERENCIA (PS, 23 sept) — Tarea 3.1.1, dueño formal: Mauricio.
# Training loop PPO completo: rollout -> GAE -> PPO updates -> logging ->
# checkpointing. Corre end-to-end sobre los envs STUB (sin ABIDES-Gym real),
# así que las recompensas son siempre 0.0 y las curvas de retorno van a
# salir planas -- eso es esperado, no un bug de este código ni del logging
# (ver src/analysis/convergence_analysis.py, que sí está probado con datos
# sintéticos). Sirve para validar que el mecanismo (formas de tensores,
# GAE, clipping, logging, checkpoints) es correcto ANTES de que ABIDES-Gym
# esté integrado y produzca recompensas reales.
#
# Mauricio: reemplazar maestro_env / ejecutor_envs por versiones con ABIDES
# integrado cuando estén listas; el resto del loop no debería necesitar
# cambios estructurales.
# ============================================================

def collect_rollout(env, actor_critic, buffer, n_steps, device, action_adapter=None):
    """Recolecta `n_steps` transiciones de un entorno de acción discreta
    única (Categorical) en `buffer`, reseteando el episodio cuando termina.

    Args:
        action_adapter: función opcional `flat_idx -> acción que espera
            env.step()`. None para el Maestro (env.step acepta el índice
            escalar directo); `ejecutor_action_space.decode_flat` para los
            Ejecutores (env.step espera un array MultiDiscrete de 3 valores).

    Returns:
        lista de retornos de los episodios que se completaron durante la
        recolección (puede quedar vacía si ningún episodio terminó).
    """
    obs, _ = env.reset()
    ep_return = 0.0
    completed_returns = []

    for _ in range(n_steps):
        obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            action, log_prob, _, value = actor_critic.get_action_and_value(obs_t)
        action_item = int(action.item())

        action_for_env = action_adapter(action_item) if action_adapter is not None else action_item
        next_obs, reward, done, truncated, info = env.step(action_for_env)

        buffer.add(obs, action_item, log_prob.item(), reward, value.item(), done or truncated)
        ep_return += reward
        obs = next_obs

        if done or truncated:
            completed_returns.append(ep_return)
            ep_return = 0.0
            obs, _ = env.reset()

    return completed_returns


ejecutor_action_space = ejecutor_envs[0]._action_space_wrapper  # decode_flat es el mismo para los 3
GAMMA_MAESTRO = config['gamma']                      # 0.99 (MDP_M, Título I sección 4)
GAMMA_EJECUTOR = config.get('gamma_ejecutor', 0.95)  # 0.95 (MDP_E, Título I sección 5)

global_step = 0
print(f"Iniciando training loop (borrador) hasta total_timesteps={config['total_timesteps']}...")

pbar = tqdm(total=config['total_timesteps'])
while global_step < config['total_timesteps']:
    # --- 1. Rollout: Maestro ---
    maestro_buffer.clear()
    maestro_ep_returns = collect_rollout(
        maestro_env, maestro_actor_critic, maestro_buffer, config['rollout_steps'], device
    )

    # --- 1b. Rollout: los 3 Ejecutores ---
    ejecutor_ep_returns = []
    for i in range(3):
        ejecutor_buffers[i].clear()
        returns_i = collect_rollout(
            ejecutor_envs[i], ejecutor_actor_critics[i], ejecutor_buffers[i],
            config['rollout_steps'], device, action_adapter=ejecutor_action_space.decode_flat
        )
        ejecutor_ep_returns.append(returns_i)

    # --- 2. Compute advantages (GAE) ---
    maestro_buffer.compute_returns_advantages(GAMMA_MAESTRO, config['gae_lambda'])
    for i in range(3):
        ejecutor_buffers[i].compute_returns_advantages(GAMMA_EJECUTOR, config['gae_lambda'])

    # --- 3. PPO updates ---
    maestro_losses = run_ppo_epochs(
        maestro_actor_critic, maestro_optimizer, maestro_buffer,
        config['epochs_per_update'], config['batch_size'],
        config['clip_ratio'], config['entropy_coef'], config['value_coef'],
    )
    ejecutor_losses = [
        run_ppo_epochs(
            ejecutor_actor_critics[i], ejecutor_optimizers[i], ejecutor_buffers[i],
            config['epochs_per_update'], config['batch_size'],
            config['clip_ratio'], config['entropy_coef'], config['value_coef'],
        )
        for i in range(3)
    ]

    global_step += config['rollout_steps']
    pbar.update(config['rollout_steps'])

    # --- 4. Logging (cada rollout, no cada eval_interval -- el rollout ya es la unidad
    #        natural de actualización de PPO; ajustar si se prefiere una cadencia distinta) ---
    R_m = float(np.mean(maestro_ep_returns)) if maestro_ep_returns else 0.0
    Rs_e = [float(np.mean(r)) if r else 0.0 for r in ejecutor_ep_returns]
    log_metrics(global_step, R_m, maestro_losses, Rs_e, ejecutor_losses)

    # --- 5. Checkpointing ---
    if global_step % config['checkpoint_interval'] < config['rollout_steps']:
        save_checkpoint(global_step)

pbar.close()
print(f"\n✓ Training loop (borrador) completado — {global_step} timesteps.")
print("  Recuerda: con los envs stub (sin ABIDES real) las recompensas son 0.0,")
print("  así que las curvas de retorno saldrán planas. Eso es esperado.")


---

**Estado**: Skeleton lista

**Bloqueadores**: Redes de Mauricio, ABIDES integration

**Próxima reunión**: Coordinar con Mauricio/Benjamín para integración